# APTOS 2019 - Model Training & Baseline Comparison

**Task 1:** Train 4 different deep learning architectures using transfer learning and compare performance.

**Models:**
1. ResNet50
2. EfficientNetB3
3. DenseNet121
4. InceptionV3

**Approach:** Pretrained ImageNet weights → freeze base → custom classification head → train → evaluate

**Primary Metric:** Quadratic Weighted Kappa (QWK) — standard for ordinal DR grading

## 1.0 Import Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import (
    ResNet50, EfficientNetB3, DenseNet121, InceptionV3
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import (
    classification_report, confusion_matrix, cohen_kappa_score
)
import warnings
warnings.filterwarnings('ignore')

SEED = 77
tf.random.set_seed(SEED)
np.random.seed(SEED)

print(f'TensorFlow version: {tf.__version__}')
print(f'GPUs available: {len(tf.config.list_physical_devices("GPU"))}')

## 2.0 Load Processed Data & Configure Generators

### ⚙️ Environment Setup (Local / Google Colab)

This cell auto-detects whether you're running locally (Windows) or on Google Colab, and sets paths accordingly.

In [ ]:
# Auto-detect environment: Local (Windows) vs Google Colab
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # === GOOGLE COLAB SETUP ===
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Install keras-tuner if needed (Colab may not have it)
    !pip install keras-tuner -q
    
    # Paths on Google Drive
    BASE_DIR = '/content/drive/MyDrive/APTOS_data'
    print(f'Running on Google Colab')
    print(f'Base directory: {BASE_DIR}')
else:
    # === LOCAL SETUP (Windows) ===
    BASE_DIR = r'C:\Users\Owent\Desktop\ODL_Official_assignment\dataScience_workflow'
    print(f'Running locally')
    print(f'Base directory: {BASE_DIR}')

# Common paths (work for both environments)
PROCESSED_DIR = os.path.join(BASE_DIR, 'data', 'processed') if not IN_COLAB else os.path.join(BASE_DIR, 'processed')
TRAIN_IMG_DIR = os.path.join(PROCESSED_DIR, 'train_images')
VAL_IMG_DIR = os.path.join(PROCESSED_DIR, 'val_images')

print(f'Processed dir: {PROCESSED_DIR}')
print(f'Train images: {TRAIN_IMG_DIR}')
print(f'Val images: {VAL_IMG_DIR}')

In [ ]:
# Load split CSVs
train_df = pd.read_csv(os.path.join(PROCESSED_DIR, 'train_split.csv'))
val_df = pd.read_csv(os.path.join(PROCESSED_DIR, 'val_split.csv'))

# Add file paths
train_df['file_path'] = train_df['id_code'].apply(lambda x: os.path.join(TRAIN_IMG_DIR, f'{x}.png'))
val_df['file_path'] = val_df['id_code'].apply(lambda x: os.path.join(VAL_IMG_DIR, f'{x}.png'))
train_df['diagnosis'] = train_df['diagnosis'].astype(str)
val_df['diagnosis'] = val_df['diagnosis'].astype(str)

# Load class weights
class_weights_df = pd.read_csv(os.path.join(PROCESSED_DIR, 'class_weights.csv'))
class_weight_dict = dict(zip(class_weights_df['class'].astype(int), class_weights_df['weight']))

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 5
EPOCHS = 20

print(f'Training samples: {len(train_df)}')
print(f'Validation samples: {len(val_df)}')
print(f'Class weights: {class_weight_dict}')

### 2.1 Create Data Generators with Augmentation

In [ ]:
# Training generator WITH augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    horizontal_flip=True,
    vertical_flip=True,
    rotation_range=30,
    zoom_range=0.15,
    brightness_range=[0.8, 1.2],
    fill_mode='constant',
    cval=0
)

# Validation generator WITHOUT augmentation (only rescale)
val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col='file_path',
    y_col='diagnosis',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True,
    seed=SEED
)

val_generator = val_datagen.flow_from_dataframe(
    dataframe=val_df,
    x_col='file_path',
    y_col='diagnosis',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print(f'Training batches per epoch: {len(train_generator)}')
print(f'Validation batches: {len(val_generator)}')

---
## 3.0 Define Model Architecture

We use a common architecture pattern for all 4 models:
1. Pretrained base (ImageNet weights) — feature extractor
2. Global Average Pooling — reduces spatial dimensions
3. Dense(512) + BatchNorm + Dropout — learns task-specific features
4. Dense(5, softmax) — 5-class DR classification

In [ ]:
def build_model(base_model_class, input_shape=(IMG_SIZE, IMG_SIZE, 3), name='model'):
    """
    Build a transfer learning model with the given pretrained base.
    Base layers are frozen; only the classification head is trainable.
    """
    # Handle InceptionV3 which requires min 75x75 input
    base = base_model_class(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )
    
    # Freeze all base layers (transfer learning)
    base.trainable = False
    
    # Build classification head
    x = base.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
    
    model = Model(inputs=base.input, outputs=outputs, name=name)
    
    model.compile(
        optimizer=Adam(learning_rate=1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

print('Model builder function defined.')

### 3.1 Define Training & Evaluation Function

In [ ]:
def train_and_evaluate(model, model_name, train_gen, val_gen, epochs=EPOCHS):
    """
    Train a model and evaluate it. Returns history and metrics.
    """
    print(f'\n{"="*60}')
    print(f'Training: {model_name}')
    print(f'Trainable parameters: {model.count_params():,}')
    print(f'{"="*60}')
    
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
    ]
    
    history = model.fit(
        train_gen,
        epochs=epochs,
        validation_data=val_gen,
        class_weight=class_weight_dict,
        callbacks=callbacks,
        verbose=1
    )
    
    # Evaluate
    val_gen.reset()
    y_pred_probs = model.predict(val_gen, verbose=0)
    y_pred = np.argmax(y_pred_probs, axis=1)
    y_true = val_gen.classes
    
    # Metrics
    accuracy = np.mean(y_pred == y_true)
    kappa = cohen_kappa_score(y_true, y_pred, weights='quadratic')
    
    print(f'\n--- Results for {model_name} ---')
    print(f'Validation Accuracy: {accuracy:.4f}')
    print(f'Quadratic Weighted Kappa: {kappa:.4f}')
    print(f'\nClassification Report:')
    print(classification_report(y_true, y_pred, target_names=['No DR','Mild','Moderate','Severe','Proliferative']))
    
    return {
        'model_name': model_name,
        'history': history,
        'accuracy': accuracy,
        'kappa': kappa,
        'y_pred': y_pred,
        'y_true': y_true
    }

print('Training function defined.')

---
## 4.0 Train All 4 Models

Each model is trained with:
- Frozen pretrained base (ImageNet transfer learning)
- Custom classification head
- Class weights to handle imbalance
- Early stopping + learning rate reduction callbacks

### 4.1 ResNet50

In [ ]:
model_resnet = build_model(ResNet50, name='ResNet50')
model_resnet.summary()
results_resnet = train_and_evaluate(model_resnet, 'ResNet50', train_generator, val_generator)

### 4.2 EfficientNetB3

In [ ]:
model_effnet = build_model(EfficientNetB3, name='EfficientNetB3')
results_effnet = train_and_evaluate(model_effnet, 'EfficientNetB3', train_generator, val_generator)

### 4.3 DenseNet121

In [ ]:
model_densenet = build_model(DenseNet121, name='DenseNet121')
results_densenet = train_and_evaluate(model_densenet, 'DenseNet121', train_generator, val_generator)

### 4.4 InceptionV3

In [ ]:
model_inception = build_model(InceptionV3, name='InceptionV3')
results_inception = train_and_evaluate(model_inception, 'InceptionV3', train_generator, val_generator)

---
## 5.0 Model Comparison

Compare all 4 models on validation set performance.

In [ ]:
# Compile results into comparison table
all_results = [results_resnet, results_effnet, results_densenet, results_inception]

comparison_df = pd.DataFrame({
    'Model': [r['model_name'] for r in all_results],
    'Val Accuracy': [r['accuracy'] for r in all_results],
    'Quadratic Weighted Kappa': [r['kappa'] for r in all_results]
}).sort_values('Quadratic Weighted Kappa', ascending=False)

print('='*60)
print('MODEL COMPARISON (Sorted by QWK)')
print('='*60)
print(comparison_df.to_string(index=False))
print('='*60)

best_model_name = comparison_df.iloc[0]['Model']
print(f'\n🏆 Best performing model: {best_model_name}')
print(f'   This model will be used for hyperparameter tuning in Task 2.')

### 5.1 Training History Comparison

In [ ]:
# Plot training curves for all models
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for idx, result in enumerate(all_results):
    row, col = idx // 2, idx % 2
    h = result['history']
    axes[row, col].plot(h.history['accuracy'], label='Train Acc')
    axes[row, col].plot(h.history['val_accuracy'], label='Val Acc')
    axes[row, col].plot(h.history['loss'], label='Train Loss', linestyle='--')
    axes[row, col].plot(h.history['val_loss'], label='Val Loss', linestyle='--')
    axes[row, col].set_title(result['model_name'], fontsize=13)
    axes[row, col].set_xlabel('Epoch')
    axes[row, col].legend()
    axes[row, col].grid(True, alpha=0.3)

plt.suptitle('Training History - All Models', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

### 5.2 Confusion Matrices

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
class_labels = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']

for idx, result in enumerate(all_results):
    row, col = idx // 2, idx % 2
    cm = confusion_matrix(result['y_true'], result['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[row, col],
                xticklabels=class_labels, yticklabels=class_labels)
    axes[row, col].set_title(f"{result['model_name']} (QWK={result['kappa']:.3f})", fontsize=12)
    axes[row, col].set_xlabel('Predicted')
    axes[row, col].set_ylabel('Actual')

plt.suptitle('Confusion Matrices - All Models', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

### 5.3 Performance Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(all_results))
width = 0.35

bars1 = ax.bar(x - width/2, [r['accuracy'] for r in all_results], width, label='Accuracy', color='steelblue')
bars2 = ax.bar(x + width/2, [r['kappa'] for r in all_results], width, label='QWK', color='coral')

ax.set_xlabel('Model')
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels([r['model_name'] for r in all_results])
ax.legend()
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)

# Add value labels
for bar in bars1 + bars2:
    h = bar.get_height()
    ax.annotate(f'{h:.3f}', xy=(bar.get_x() + bar.get_width()/2, h),
                xytext=(0, 3), textcoords='offset points', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

---
## 6.0 Task 1 Summary

### Model Architectures Used:

| Model | Architecture Type | Key Feature | Parameters |
|-------|------------------|-------------|------------|
| ResNet50 | Residual Network | Skip connections prevent vanishing gradients | ~25.6M |
| EfficientNetB3 | Compound Scaling | Balanced depth/width/resolution scaling | ~12.2M |
| DenseNet121 | Dense Connections | Feature reuse across all layers | ~8.0M |
| InceptionV3 | Multi-scale | Parallel convolutions with different kernel sizes | ~23.8M |

### Training Configuration:
- **Transfer Learning:** ImageNet pretrained weights (base frozen)
- **Optimizer:** Adam (lr=1e-3)
- **Loss:** Categorical Crossentropy
- **Class Weights:** Applied to handle imbalance
- **Callbacks:** EarlyStopping (patience=5), ReduceLROnPlateau
- **Augmentation:** Flip, rotation, zoom, brightness (training only)

### Next Step:
→ Task 2: Hyperparameter tuning on the best model using Keras Tuner